<a href="https://colab.research.google.com/github/AyaAlharthi/diamond-price-prediction/blob/main/taxi_trips_sql_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taxi Trips SQL Analysis
**Goal:** answer business questions about revenue, demand and tipping using SQL.

**Data:** NYC taxi trips (seaborn `taxis` dataset, 6,383 trips after removing rows with missing zones).
**Setup:** the data is loaded into a SQLite database (`trips` table) and queried with SQL.

In [1]:
import sqlite3, pandas as pd, seaborn as sns

df = sns.load_dataset("taxis").dropna(subset=["pickup_zone", "dropoff_zone"])
df["pickup"] = df["pickup"].astype(str)
df["dropoff"] = df["dropoff"].astype(str)

conn = sqlite3.connect("taxis.db")
df.to_sql("trips", conn, if_exists="replace", index=False)

def q(sql):
    return pd.read_sql_query(sql, conn)

q("SELECT COUNT(*) AS total_trips FROM trips")

,total_trips
0,6383


# Taxi Trips SQL Analysis
**Goal:** answer business questions about revenue, demand and tipping using SQL.

**Data:** NYC taxi trips (seaborn `taxis` dataset, 6,383 trips after removing rows with missing zones).
**Setup:** the data is loaded into a SQLite database (`trips` table) and queried with SQL.

In [2]:
# 1. Revenue and average fare by payment type
q("""SELECT payment, COUNT(*) AS trips, ROUND(SUM(total),2) AS revenue, ROUND(AVG(fare),2) AS avg_fare
     FROM trips GROUP BY payment ORDER BY revenue DESC""")

,payment,trips,revenue,avg_fare
0,credit card,4546,90362.15,13.53
1,cash,1795,25743.23,11.26
2,None,42,582.62,10.69


**Finding:** Card payments generate about 77% of total revenue ($90,362 of $116,688) and have a higher average fare ($13.53 vs $11.26 for cash). 42 trips have no payment type recorded, which is a data quality issue worth flagging.

## Query 2: Top 10 pickup zones by revenue
**Question:** Which zones earn the most?
**Technique:** `GROUP BY`, `ORDER BY`, `LIMIT`

In [4]:
# 2. Top 10 pickup zones by revenue
q("""SELECT pickup_zone, COUNT(*) AS trips, ROUND(SUM(total),2) AS revenue
     FROM trips GROUP BY pickup_zone ORDER BY revenue DESC LIMIT 10""")

,pickup_zone,trips,revenue
0,JFK Airport,146,7967.86
1,LaGuardia Airport,144,6016.90
2,Midtown Center,229,4193.08
3,Penn Station/Madison Sq West,210,3652.63
4,Times Sq/Theatre District,183,3413.38
5,Midtown East,197,3374.93
6,Clinton East,208,3304.37
7,Union Sq,180,3068.70
8,Lincoln Square East,177,3044.48
9,Upper East Side South,210,2959.39


**Finding:** JFK and LaGuardia airports are the top two zones by revenue. Together they are only 290 trips (about 4.5% of all trips) but bring in about 12% of total revenue.

## Query 3: Busiest hours
**Question:** When is demand highest?
**Technique:** `strftime`, `GROUP BY`

In [5]:
# 3. Busiest hours of the day
q("""SELECT strftime('%H', pickup) AS hour, COUNT(*) AS trips
     FROM trips GROUP BY hour ORDER BY trips DESC LIMIT 5""")

,hour,trips
0,18,417
1,19,403
2,17,385
3,20,365
4,14,356


**Finding:** Demand peaks in the evening: 18:00 is the busiest hour (417 trips), followed by 19:00 and 17:00.

## Query 4: Average tip percentage by borough
**Question:** Where do passengers tip the most? (card payments only, because cash tips are not recorded)
**Technique:** calculated column, `WHERE`, `GROUP BY`

In [6]:
# 4. Average tip percentage by borough (card payments only)
q("""SELECT pickup_borough, ROUND(AVG(tip*100.0/fare),1) AS avg_tip_pct
     FROM trips WHERE payment='credit card' AND fare>0
     GROUP BY pickup_borough ORDER BY avg_tip_pct DESC""")

,pickup_borough,avg_tip_pct
0,Manhattan,25.6
1,Queens,17.9
2,Brooklyn,11.7
3,Bronx,1.3


**Finding:** Manhattan has the highest average tip (25.6%), followed by Queens (17.9%) and Brooklyn (11.7%). The Bronx shows 1.3%, but its zones have very few trips (the biggest has 9), so this figure is based on a small sample and should not be generalised.

In [ ]:
## Query 5: Zones with above-average fares
**Question:** Which zones have fares higher than the overall average?
**Technique:** subquery, `HAVING` (only zones with at least 20 trips, to avoid tiny samples)

In [7]:
# 5. Zones with above-average fare (subquery)
q("""SELECT pickup_zone, ROUND(AVG(fare),2) AS avg_fare, COUNT(*) AS trips
     FROM trips GROUP BY pickup_zone
     HAVING trips >= 20 AND AVG(fare) > (SELECT AVG(fare) FROM trips)
     ORDER BY avg_fare DESC LIMIT 10""")

,pickup_zone,avg_fare,trips
0,JFK Airport,43.52,146
1,LaGuardia Airport,29.35,144
2,Long Island City/Hunters Point,16.45,20
3,World Trade Center,16.06,24
4,Downtown Brooklyn/MetroTech,15.85,37
5,Battery Park City,15.85,43
6,Financial District North,15.69,35
7,Washington Heights South,15.23,32
8,Bloomingdale,13.50,20
9,Brooklyn Heights,13.28,34


**Finding:** JFK ($43.52) and LaGuardia ($29.35) stand far above every other zone. The remaining zones are only slightly above the overall average, so airport pickups are the clear high-fare segment.

## Query 6: Top 3 pickup zones in each borough
**Question:** What are the busiest zones inside each borough?
**Technique:** window function `RANK() OVER (PARTITION BY ...)`

In [8]:
# 6. Rank pickup zones within each borough (window function)
q("""SELECT * FROM (
       SELECT pickup_borough, pickup_zone, COUNT(*) AS trips,
              RANK() OVER (PARTITION BY pickup_borough ORDER BY COUNT(*) DESC) AS rnk
       FROM trips GROUP BY pickup_borough, pickup_zone)
     WHERE rnk <= 3""")

,pickup_borough,pickup_zone,trips,rnk
0,Bronx,East Concourse/Concourse Village,9,1
1,Bronx,Parkchester,8,2
2,Bronx,Mott Haven/Port Morris,8,2
3,Brooklyn,Fort Greene,39,1
4,Brooklyn,Downtown Brooklyn/MetroTech,37,2
5,Brooklyn,Brooklyn Heights,34,3
6,Manhattan,Midtown Center,229,1
7,Manhattan,Upper East Side South,210,2
8,Manhattan,Penn Station/Madison Sq West,210,2
9,Queens,JFK Airport,146,1


**Finding:** Midtown Center leads Manhattan (229 trips), Fort Greene leads Brooklyn (39), JFK leads Queens (146) and East Concourse leads the Bronx (9). `RANK` keeps ties, which is why two zones share rank 2 in Manhattan and in the Bronx.

## Query 7: Fare by trip distance
**Question:** How does fare change with trip length?
**Technique:** `CASE WHEN` to create distance buckets

In [9]:
# 7. Trip distance buckets (CASE WHEN)
q("""SELECT CASE WHEN distance < 1 THEN 'Short (<1 mi)'
                WHEN distance < 5 THEN 'Medium (1-5 mi)'
                ELSE 'Long (5+ mi)' END AS bucket,
            COUNT(*) AS trips, ROUND(AVG(fare),2) AS avg_fare
     FROM trips GROUP BY bucket""")

,bucket,trips,avg_fare
0,Long (5+ mi),952,33.12
1,Medium (1-5 mi),3828,10.84
2,Short (<1 mi),1603,5.70


**Finding:** Medium trips (1-5 miles) are the majority (3,828 trips), while long trips (5+ miles) are only about 15% of trips but earn an average fare of $33.12, roughly 6 times a short trip ($5.70).

## Summary
- Card payments drive about 77% of revenue.
- Airport pickups are about 4.5% of trips but about 12% of revenue.
- Demand peaks in the evening around 18:00.
- Long trips are rare but earn about 6x the fare of short trips.
- Limitations: 42 trips have no payment type, and borough-level tip rates for the Bronx rely on very few trips.